# Phase 1: creating a db of article metadata

The JSON metadata file is updated weekly by ArXiv directly on Kaggle: https://www.kaggle.com/datasets/Cornell-University/arxiv

In [1]:
import duckdb
import json

In [ ]:
# Read the first line of the JSON file we downloaded to get a better understanding of its structure
with open('data/raw/arxiv-metadata-oai-snapshot.json', 'r') as json_file:
    data = json.loads(json_file.readline())

print("Keys:", list(data.keys()))
print("\nSample values:")
data

Keys: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed']

Sample values:


{'id': '0704.0001',
 'submitter': 'Pavel Nadolsky',
 'authors': "C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan",
 'title': 'Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies',
 'comments': '37 pages, 15 figures; published version',
 'journal-ref': 'Phys.Rev.D76:013009,2007',
 'doi': '10.1103/PhysRevD.76.013009',
 'report-no': 'ANL-HEP-PR-07-12',
 'categories': 'hep-ph',
 'license': None,
 'abstract': '  A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massive photon pairs at hadron colliders. All\nnext-to-leading order perturbative contributions from quark-antiquark,\ngluon-(anti)quark, and gluon-gluon subprocesses are included, as well as\nall-orders resummation of initial-state gluon radiation valid at\nnext-to-next-to-leading logarithmic accuracy. The region of phase space is\nspecified in which the calculation is most reliable. Good agreement is\ndemonstrated with data from th

In [3]:
with open('data/raw/arxiv-metadata-oai-snapshot.json', 'r') as json_file:
    for i in range(5):  # Check first 5 papers
        line = json_file.readline()
        paper = json.loads(line)
        print(f"Paper {i+1} comments:", paper.get('comments', 'No comments'))
        print(f"Paper {i+1} abstract:", paper.get('abstract', 'No abstract')[:100] + "...")

Paper 1 comments: 37 pages, 15 figures; published version
Paper 1 abstract:   A fully differential calculation in perturbative quantum chromodynamics is
presented for the produ...
Paper 2 comments: To appear in Graphs and Combinatorics
Paper 2 abstract:   We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use
it obtain a character...
Paper 3 comments: 23 pages, 3 figures
Paper 3 abstract:   The evolution of Earth-Moon system is described by the dark matter field
fluid model proposed in t...
Paper 4 comments: 11 pages
Paper 4 abstract:   We show that a determinant of Stirling cycle numbers counts unlabeled acyclic
single-source automa...
Paper 5 comments: None
Paper 5 abstract:   In this paper we show how to compute the $\Lambda_{\alpha}$ norm, $\alpha\ge
0$, using the dyadic ...


## DuckDB creation

In [6]:
conn = duckdb.connect('data/arxiv_metadata.duckdb')

In [7]:
conn.execute("""
    CREATE TABLE staging AS
    SELECT * FROM read_json_auto('data/raw/arxiv-metadata-oai-snapshot.json');
""")

print(conn.execute("DESCRIBE staging").fetchdf())

print(conn.execute("SELECT * FROM staging LIMIT 1").fetchdf())

       column_name                                   column_type null   key  \
0               id                                       VARCHAR  YES  None   
1        submitter                                       VARCHAR  YES  None   
2          authors                                       VARCHAR  YES  None   
3            title                                       VARCHAR  YES  None   
4         comments                                       VARCHAR  YES  None   
5      journal-ref                                       VARCHAR  YES  None   
6              doi                                       VARCHAR  YES  None   
7        report-no                                       VARCHAR  YES  None   
8       categories                                       VARCHAR  YES  None   
9          license                                       VARCHAR  YES  None   
10        abstract                                       VARCHAR  YES  None   
11        versions  STRUCT("version" VARCHAR, create